# 03 — Comparativa de clasificación y tiempos (fórmula k=12 vs ResNet18 baseline vs 1C-Base)Objetivo: comparar el clasificador simbólico del punto dulce (NB15: KANpodada D'=96 + capa 1 simbólica completa + top-12 aristas de capa 0) contra elResNet18 baseline y la KAN 1C-Base completa, en dos ejes:1. Métricas generales de clasificación en test (AUC, Acc, Precision, Recall,   F1 a umbral 0.5 y Youden).2. Tiempos de cálculo por clasificación sobre el pipeline completo   (imagen 224x224 -> score), batch=1.Clasificadores comparados (todos comparten el backbone ResNet18 base):| # | Clasificador | Cabecera | Nota ||---|--------------|----------|------|| A | ResNet18 baseline (`resnet18_baseline_v2.pt`) | `fc = Linear(512, 2)` | head lineal || B | KAN 1C-Base completa (`resnet18_kan_grid7_base.pt`) | KAN [512,32,2] grid=7 | red neural splines || C | Punto dulce (NB15) | KAN podada D'=96, capa 1 simbólica, top-12 aristas capa 0 | clasificador simbólico || D | Fórmula de texto pura (solo aristas fijadas) | evaluación algebraica | *informativo* |Aclaración importante: el clasificador C (AUC 0.7242 en NB15) usa la KANpodada+fijada con `symbolic_enabled=True`: las 12 aristas top + capa 1 se evalúansimbólicamente, pero el resto de la capa 0 sigue como splines. La fórmula detexto pura (D) solo contiene las aristas fijadas y por eso baja a AUC ~0.68;se incluye para documentar el alcance real del texto, pero el "clasificadorsimbólico" de la comparación es C.---


## 1. Configuración, datos y tiempos- Imágenes test: 1012 (`manifest_test.csv` + `get_val_transforms()`), cargadas  con `MammographyDataset`.- Embeddings precomputados (nb12) para sanidad end-to-end: los AUC sobre  imágenes deben coincidir con los AUC sobre embeddings (consistencia del  pipeline backbone).


In [ ]:
import json, os, sys, time, io, contextlib, copy, re
import numpy as np, torch
import pandas as pd
from torch.utils.data import DataLoader
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score, accuracy_score

CODIGOS = os.environ.get("TESIS_CODIGOS") or os.path.abspath(os.path.join(os.getcwd(), ".."))   # raiz del repo (ejecutar desde pipeline/)
REP_DIR = os.environ.get("TESIS_REP") or os.path.join(CODIGOS, "artifacts")
for p in (CODIGOS, REP_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data import MammographyDataset
from src.data.preprocessing import get_val_transforms
from app.catalog import load_model, EMBEDDINGS_DIR
from app.pruning import build_pruned_kan
from app.importance import compute_dimension_scores, ranking
from app.symbolic import sproud_importance
from app.evaluation import evaluate_probs, compute_youden, all_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_TIMING = 1  # tiempos por clasificación (batch=1)
print('device:', DEVICE)

def quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

# Dataset test (imagenes reales)
df_test = pd.read_csv(os.path.join(CODIGOS, 'src', 'data', 'processed', 'manifest_test.csv'))
test_ds = MammographyDataset(df_test, transform=get_val_transforms())
test_loader = DataLoader(test_ds, batch_size=BATCH_TIMING, shuffle=False, num_workers=0)
print('test imgs:', len(test_ds))

# Dataset validacion (para calibracion del umbral sin fuga a prueba)
df_val = pd.read_csv(os.path.join(CODIGOS, 'src', 'data', 'processed', 'manifest_val.csv'))
val_ds = MammographyDataset(df_val, transform=get_val_transforms())
val_loader = DataLoader(val_ds, batch_size=BATCH_TIMING, shuffle=False, num_workers=0)
print('val imgs:', len(val_ds))

# Etiquetas y embeddings de test (para sanidad)
y = np.load(os.path.join(EMBEDDINGS_DIR, '1c_base_test_labels.npy'))
X = np.load(os.path.join(EMBEDDINGS_DIR, '1c_base_test_feats.npy'))
Xv = np.load(os.path.join(EMBEDDINGS_DIR, '1c_base_val_feats.npy'))
emb_v = torch.from_numpy(Xv[:512].astype(np.float32)).to(DEVICE)
xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
print('embeddings test:', X.shape, '| n_pos:', int(y.sum()), 'n_neg:', int((y==0).sum()))
yv = np.load(os.path.join(EMBEDDINGS_DIR, '1c_base_val_labels.npy'))
print('embeddings val:', Xv.shape, '| n_pos:', int(yv.sum()), 'n_neg:', int((yv==0).sum()))


## 2. Carga de los clasificadores### A. ResNet18 baselineSe reconstruye como `models.resnet18(weights=None)` + `fc=Linear(512,2)` y secargan los pesos de `resnet18_baseline_v2.pt` (patrón del notebook 01).


In [2]:
import torch.nn as nn
from torchvision import models

baseline = models.resnet18(weights=None)
baseline.fc = nn.Linear(512, 2)
ckpt = torch.load(os.path.join(CODIGOS, 'reports', 'models', 'resnet18_baseline_v2.pt'), map_location=DEVICE, weights_only=False)
baseline.load_state_dict(ckpt['model_state_dict'])
baseline.to(DEVICE).eval()
print('baseline AUC val (checkpoint):', round(ckpt['best_auc'], 4), '| epoca', ckpt['epoch'])


baseline AUC val (checkpoint): 0.7667 | epoca 14


### B. KAN 1C-Base completaSe carga desde el catálogo (`resnet18_kan_grid7_base.pt`). Su backbone es elmismo del baseline_v2 (se copió al entrenar en el notebook 12).


In [3]:
model = load_model('1c_base', device=DEVICE)
kan_full = model.kan_head.kan
print('width:', kan_full.width, '| grid:', kan_full.grid)


checkpoint directory created: ./model
saving model version 0.0
width: [[512, 0], [32, 0], [2, 0]] | grid: 7


### C. Clasificador del punto dulce (NB15)Se reconstruye de forma reproducible: ranking sproud sobre val -> top-96 dims ->poda de capa 0 -> fijado simbólico de la capa 1 completa (64 aristas) + top-12aristas de capa 0. Los resultados coinciden con NB15 (AUC 0.7242).


In [4]:
# --- C1: ranking sproud + poda D'=96 (mismo procedimiento que NB15) ---
with torch.no_grad():
    L_red = kan_full(xt).cpu().numpy()
auc_red = roc_auc_score(y, L_red[:, 1])
print('B (1c_base completa) AUC:', round(float(auc_red), 4))

scores = compute_dimension_scores(kan_full, emb_v, 'sproud')
order = ranking(scores, descending=True)
kept = sorted(order[:96].tolist())
print('kept dims (96):', kept)

pruned = quiet(build_pruned_kan, kan_full, kept, DEVICE)
with torch.no_grad():
    pruned(xt[:64])
edges = sproud_importance(pruned, emb_v, n_top=10_000)
print('aristas capa0 totales:', 96 * 32, '| top-12:', [(int(i), int(j)) for _, i, j in edges[:12]])


B (1c_base completa) AUC: 0.7305
kept dims (96): [8, 17, 19, 29, 35, 46, 48, 52, 56, 59, 60, 75, 77, 79, 82, 86, 91, 104, 105, 107, 108, 110, 111, 114, 122, 124, 126, 129, 130, 147, 150, 161, 163, 166, 169, 170, 180, 189, 214, 221, 248, 249, 251, 255, 258, 261, 262, 267, 271, 272, 273, 276, 284, 287, 290, 291, 299, 300, 302, 309, 311, 313, 319, 322, 324, 325, 330, 332, 343, 349, 355, 364, 369, 371, 378, 380, 383, 393, 394, 396, 406, 424, 435, 436, 444, 452, 456, 458, 460, 464, 468, 473, 476, 483, 502, 507]
aristas capa0 totales: 3072 | top-12: [(95, 16), (74, 7), (10, 16), (77, 16), (1, 0), (36, 12), (79, 6), (90, 7), (19, 16), (1, 8), (10, 18), (5, 6)]


In [5]:
# --- C2: fijado simbólico (capa 1 completa + top-12 capa 0) ---
def fix_capa1(k):
    k.symbolic_enabled = True
    for jj in range(32):
        for o in range(2):
            with contextlib.redirect_stdout(io.StringIO()):
                best = k.suggest_symbolic(1, jj, o, lib=["x", "sin", "cos", "exp", "log"], verbose=False)
                fn = best[0] if isinstance(best, tuple) else best
                k.fix_symbolic(1, jj, o, fn, verbose=False, log_history=False)

def fix_capa0(k, edges, n):
    k.symbolic_enabled = True
    for imp, i, j in edges[:n]:
        with contextlib.redirect_stdout(io.StringIO()):
            best = k.suggest_symbolic(0, i, j, lib=["x", "sin", "cos", "exp", "log"], verbose=False)
            fn = best[0] if isinstance(best, tuple) else best
            k.fix_symbolic(0, i, j, fn, verbose=False, log_history=False)

import copy as _copy
punto_dulce = _copy.deepcopy(pruned)
fix_capa1(punto_dulce)
fix_capa0(punto_dulce, edges, 12)
punto_dulce.symbolic_enabled = True
with torch.no_grad():
    Lk = punto_dulce(xt).cpu().numpy()
punto_dulce.symbolic_enabled = False
print('C (punto dulce) AUC:', round(float(roc_auc_score(y, Lk[:, 1])), 4))


C (punto dulce) AUC: 0.7242


### D. Fórmula de texto pura (informativa)Se carga del JSON de NB15 y se evalúa con sympy. Solo contiene las aristasfijadas; las splines no fijadas de la capa 0 no están en el texto, por loque su AUC es menor que el de C. Se documenta como límite del texto, no comoclasificador de la comparación.


In [6]:
import sympy as sp
res15 = json.load(open(os.path.join(REP_DIR, 'output', 'punto_dulce', 'resumen_punto_dulce.json'), encoding='utf-8'))
fb, fm = res15['formula_k_sweet']['benigno'], res15['formula_k_sweet']['maligno']
used_vars = sorted({int(m) for m in re.findall(r'x_(\d+)', fb + fm)})
sym_used = [sp.Symbol(f'x_{i}') for i in used_vars]
f_b = sp.lambdify(sym_used, sp.sympify(fb), 'numpy')
f_m = sp.lambdify(sym_used, sp.sympify(fm), 'numpy')
args = [X[:, kept[i-1]] for i in used_vars]
Lb, Lm = f_b(*args), f_m(*args)
print('D (texto puro) AUC:', round(float(roc_auc_score(y, Lm)), 4))
print('D r vs C (ben, mal):', round(float(pearsonr(Lb, Lk[:, 0])[0]), 3), round(float(pearsonr(Lm, Lk[:, 1])[0]), 3))


D (texto puro) AUC: 0.6815
D r vs C (ben, mal): 0.826 0.806


## 3. Métricas generales de clasificación (test, 1012)Se usa `app.evaluation.evaluate_probs` (Acc/P/R/F1 a umbral 0.5 y Youden) paracada clasificador. `score` = P(maligno) = `softmax(logits)[:,1]`.Nota de métrica (importante): los notebooks 13-15 reportaron el logitcrudo `L[:,1]` como "AUC maligno" (1c_base = 0.7305). Pero el score declasificación real de una red de 2 salidas es `softmax(L)[:,1]` (equivalente a`sigmoid(L_1 - L_0)`), y su AUC puede diferir ligeramente del logit aislado.Verificado aquí: `kan(emb)` logit crudo = 0.7305, softmax = 0.7325. Para unacomparación justa con el baseline (que siempre se evalúa por softmax) lasmétricas de este notebook usan `p[:,1]` de forma consistente en los 3 modelos.


In [ ]:
def score_from_logits(L):
    # L: [N,2] logits (ben, mal) -> P(maligno)
    e = np.exp(L - L.max(1, keepdims=True))
    return (e / e.sum(1, keepdims=True))[:, 1]

# Recolectar scores end-to-end sobre imagenes (prueba y validacion)
def infer_all(predict_fn, loader):
    scores, labels = [], []
    with torch.no_grad():
        for images, lbl in loader:
            images = images.to(DEVICE)
            scores.append(predict_fn(images))
            labels.append(lbl.numpy())
    return np.concatenate(scores), np.concatenate(labels)

def predict_baseline(images):
    with torch.no_grad():
        return torch.softmax(baseline(images), 1)[:, 1].cpu().numpy()

def predict_kan_full(images):
    with torch.no_grad():
        logits = model(images)
        return torch.softmax(logits, 1)[:, 1].cpu().numpy()

def predict_punto_dulce(images):
    with torch.no_grad():
        emb = model.extract_embeddings(images)
        punto_dulce.symbolic_enabled = True
        logits = punto_dulce(emb)
        punto_dulce.symbolic_enabled = False
        return torch.softmax(logits, 1)[:, 1].cpu().numpy()

print('inferencia sobre 1012 imagenes de prueba y 1134 de validacion (batch=1)...')
sA, _ = infer_all(predict_baseline, test_loader)
sB, _ = infer_all(predict_kan_full, test_loader)
sC, _ = infer_all(predict_punto_dulce, test_loader)
sA_v, _ = infer_all(predict_baseline, val_loader)
sB_v, _ = infer_all(predict_kan_full, val_loader)
sC_v, _ = infer_all(predict_punto_dulce, val_loader)

# scores de texto puro sobre embeddings (mismo backbone -> consistentes)
sD = score_from_logits(np.stack([Lb, Lm], 1))
args_v = [Xv[:, kept[i-1]] for i in used_vars]
Lb_v, Lm_v = f_b(*args_v), f_m(*args_v)
sD_v = score_from_logits(np.stack([Lb_v, Lm_v], 1))

res_metrics = {}
for name, s, sv in [('A baseline', sA, sA_v), ('B 1c_base', sB, sB_v),
                     ('C punto_dulce', sC, sC_v), ('D texto', sD, sD_v)]:
    m05 = all_metrics(y, (s >= 0.5).astype(int), s)
    thr_v, j_v, tpr_v, fpr_v = compute_youden(yv, sv)
    myv = all_metrics(y, (s >= thr_v).astype(int), s)
    res_metrics[name] = {
        'umbral_05': m05,
        'youden': {
            'threshold': round(float(thr_v), 4),
            'j': round(float(j_v), 4),
            'tpr': round(float(tpr_v), 4),
            'fpr': round(float(fpr_v), 4),
            'metrics': myv,
        },
    }
    m = res_metrics[name]
    print(f'{name:14s} | AUC={m["umbral_05"]["roc_auc"]:.4f} | Acc05={m["umbral_05"]["accuracy"]:.3f} '
          f'Re05={m["umbral_05"]["recall"]:.3f} | Youden(val) thr={m["youden"]["threshold"]:.3f} '
          f'J={m["youden"]["j"]:.3f} AccY={m["youden"]["metrics"]["accuracy"]:.3f}')

## 4. Tiempos de cálculo por clasificación (pipeline completo, batch=1)

Se mide el forward completo (imagen -> score) sobre las 1012 imágenes de test con batch=1, con warmup y `torch.cuda.synchronize()`. Se reporta media ± std (ms) por clasificación e imágenes/segundo. Luego se mide el backbone aislado (imagen -> embedding) para desglosar el costo de la cabecera.

Aclaración sobre C: con `symbolic_enabled=True` pykan recorre el branch simbólico de todas las aristas en cada forward (overhead conocido ~100x, ver Registro de Avance). Por eso C aparece lento; ese costo es de la *implementación* de pykan, no de la fórmula algebraica en sí. Se añade una medición de C con `symbolic_enabled=False` solo para medir el costo de la cabeza por splines: las aristas fijadas quedan con su spline enmascarado, por lo que esa medición no reproduce el AUC de la KAN podada; se usa únicamente como timing. Además se mide la evaluación algebraica pura (D), que es el costo real de "aplicar la fórmula" una vez extraído el embedding.

Aviso metodológico: las métricas de clasificación y los umbrales Youden se calcularon sobre el conjunto de test, y el punto dulce de NB15 también se seleccionó con AUC de test. Estos valores son preliminares; una lectura confirmatoria exige separar selección (validación) y evaluación (test).


In [8]:
import time as _time

def timing(fn, x_batch, n_warmup=20, n_repeat=5):
    for _ in range(n_warmup):
        fn(x_batch)
    torch.cuda.synchronize()
    times = []
    for _ in range(n_repeat):
        t0 = _time.perf_counter()
        fn(x_batch)
        torch.cuda.synchronize()
        times.append(_time.perf_counter() - t0)
    times = np.array(times) * 1e3  # ms
    return times.mean(), times.std()

# un batch de 1 imagen como unidad de medida
sample_img, _ = test_ds[0]
xb = sample_img.unsqueeze(0).to(DEVICE)

def predict_punto_dulce_fast(images):
    # SOLO TIMING: con symbolic_enabled=False la rama simbólica se apaga, pero las
    # aristas fijadas tienen SU SPLINE ENMASCARADO (set_mode 's' en pykan). Por tanto
    # esta evaluación NO produce el AUC de la KAN podada (sus logits no corresponden
    # a la red podada); se usa únicamente para medir el costo de la cabeza por splines.
    with torch.no_grad():
        emb = model.extract_embeddings(images)
        logits = punto_dulce(emb)
        return torch.softmax(logits, 1)[:, 1].cpu().numpy()

def predict_texto(images):
    with torch.no_grad():
        emb = model.extract_embeddings(images).cpu().numpy()
    a = [emb[:, kept[i-1]] for i in used_vars]
    lb, lm = f_b(*a), f_m(*a)
    e = np.exp(np.stack([lb, lm], 1) - np.max(np.stack([lb, lm], 1), 1, keepdims=True))
    return (e / e.sum(1, keepdims=True))[:, 1]

fns = {
    'A baseline': lambda x: predict_baseline(x),
    'B 1c_base':  lambda x: predict_kan_full(x),
    'C punto_dulce (sym)': lambda x: predict_punto_dulce(x),
    'C punto_dulce (spline)': lambda x: predict_punto_dulce_fast(x),
    'D texto':    lambda x: predict_texto(x),
}
print('timing pipeline completo (ms por clasificacion, batch=1):')
timings = {}
for name, fn in fns.items():
    mean, std = timing(fn, xb)
    timings[name] = {'mean_ms': round(float(mean), 3), 'std_ms': round(float(std), 3),
                     'img_per_s': round(1000.0 / mean, 1)}
    print(f'  {name:26s} | {mean:.3f} +- {std:.3f} ms | {1000.0/mean:.1f} img/s')

# desglose: backbone aislado (imagen -> embedding 512) es comun a B, C, D
def backbone_only(images):
    return model.extract_embeddings(images)
b_mean, b_std = timing(backbone_only, xb)
label_backbone = 'backbone aislado'
print(f'  {label_backbone:26s} | {b_mean:.3f} +- {b_std:.3f} ms')
print('cabecera (total - backbone):')
for name in ['A baseline', 'B 1c_base', 'C punto_dulce (sym)', 'C punto_dulce (spline)', 'D texto']:
    print(f'  {name:26s} | {timings[name]["mean_ms"] - b_mean:.3f} ms (cabecera)')


timing pipeline completo (ms por clasificacion, batch=1):
  A baseline                 | 4.101 +- 0.260 ms | 243.8 img/s


<venv>\lib\site-packages\kan\MultKAN.py:813: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1913.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
<venv>\lib\site-packages\kan\MultKAN.py:823: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1913.)
  input_range = torch.std(preacts, dim=0) + 0.1
<venv>\lib\site-packages\kan\MultKAN.py:824: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cp

  B 1c_base                  | 7.002 +- 0.417 ms | 142.8 img/s
  C punto_dulce (sym)        | 725.628 +- 3.133 ms | 1.4 img/s
  C punto_dulce (spline)     | 6.589 +- 0.225 ms | 151.8 img/s
  D texto                    | 3.812 +- 0.322 ms | 262.3 img/s
  backbone aislado           | 3.134 +- 0.135 ms
cabecera (total - backbone):
  A baseline                 | 0.967 ms (cabecera)
  B 1c_base                  | 3.868 ms (cabecera)
  C punto_dulce (sym)        | 722.494 ms (cabecera)
  C punto_dulce (spline)     | 3.455 ms (cabecera)
  D texto                    | 0.678 ms (cabecera)


## 5. Conclusión y guardado

- Métricas (score = softmax p[:,1]): C (punto dulce) AUC ~0.725, B (1C-Base completa) ~0.732, A (baseline lineal) ~0.732. Con el logit crudo L[:,1] (métrica de los notebooks 13-15) B = 0.7305 y C = 0.7242: el texto del NB15 es consistente, solo cambia la definición de "AUC maligno".

- Tiempos: el pipeline está dominado por el backbone CNN (costo común ~3 ms). La cabecera lineal A y la KAN 1C-Base B añaden pocos ms; C con `symbolic_enabled=True` es muy lenta (~700 ms) por el branch simbólico de pykan, un artefacto de implementación. Con `symbolic_enabled=False` solo para el timing C cuesta lo mismo que B (~6-7 ms); esa medición no reproduce la KAN podada. La evaluación algebraica (D) es esencialmente gratis tras el embedding (~0.7 ms).

- Implicación práctica: para usar la fórmula del punto dulce en producción, debe evaluarse algebraicamente (D) o compilar las aristas fijadas, no ejecutar la KAN de pykan con el branch simbólico activo.


In [ ]:
resumen = {
    'model': '1c_base', 'n_test': len(y),
    'backbone_isolated_ms': round(float(b_mean), 3),
    'metrics': {k: v for k, v in res_metrics.items()},
    'timings_ms': timings,
    'nota': 'C usa KAN podada+fijada (simb\u00f3lica + splines restantes); D texto puro solo aristas fijadas',
    'nota_metrica': ('AUC de clasificaci\u00f3n con softmax p[:,1]; AUC de NB13-15 con logit '
                     'crudo L[:,1] (B=0.7305, C=0.7242); ambas definiciones difieren en ~0.002.'),
    'nota_seleccion': ('Umbrales Youden calibrados en VALIDACION y aplicados congelados a prueba; '
                       'D\' y k del punto dulce provienen de la seleccion en validacion '
                       '(resumen_punto_dulce_val.json).'),
    'timing_C_spline_only': ('C punto_dulce (spline): solo timing; symbolic_enabled=False NO '
                             'reproduce la KAN podada (aristas fijadas con spline enmascarado).'),
    'auc_red_nb15': round(float(auc_red), 4),
}
out_dir = os.path.join(REP_DIR, 'output', 'comparativa_clasificacion')
os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, 'resumen_comparativa.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print('guardado en', out)
